In [ ]:
from pathlib import Path
import sys

# =========================
# Main parameters to modify
# =========================
REPO_ROOT = Path("..").resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CSV_PATH = REPO_ROOT / "data" / "HAM10000" / "ham_test_cam_all_with_masks_grouped_by_class.csv"
IMG_DIR = REPO_ROOT / "data" / "HAM10000"

CHECKPOINT = REPO_ROOT / "external" / "checkpoints3" / "checkpoint-best-HA075.pth"
CHECKPOINT_MODEL_TYPE = "panderm"
CLASS_PRESET = "ham"
CLASS_NAMES = None

TARGET_BLOCK_INDEX = -4
COMPARE_MODE = "gt_topk_non_target"
TOPK_COMPARE = 3
ALPHA = 0.8

# Start small for smoke test. Set to 0 for full CSV.
NUM_SAMPLES = 0
START_INDEX = 0

METHODS = [
    "gradcam_target",
    "gradcam_diff",
    "finercam",
]

INCLUDE_RANDOM_BASELINE = True

# Incremental curve settings
PERTURBATION_RATIOS = [i / 100 for i in range(0, 101, 5)]
PERTURBATION_TYPES = [
    "darken",
    "desaturate",
    "gaussian_noise",
]

DARKEN_FACTOR = 0.45
NOISE_STD = 0.05
RANDOM_SEED = 42

DEVICE = None

OUT_ROOT = REPO_ROOT / "outputs" / f"soft_perturbation_curves_ha075_block{TARGET_BLOCK_INDEX}"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

SAVE_CAM_CACHE = True
CAM_CACHE_DIR = OUT_ROOT / "cam_cache"
CAM_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("CSV_PATH:", CSV_PATH)
print("CHECKPOINT:", CHECKPOINT)
print("OUT_ROOT:", OUT_ROOT)
print("CAM_CACHE_DIR:", CAM_CACHE_DIR)

In [ ]:
import json
import warnings
from types import SimpleNamespace

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from scripts.generate_finer_cam_panderm import (
    PanDermCAMWrapper,
    build_class_maps,
    load_panderm_finetuned_model,
    resolve_class_names,
    vit_reshape_transform,
)

from src.cam.diff_cam import compute_cam_bundle

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
def get_device(device_arg=None):
    if device_arg is not None:
        return device_arg
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def build_eval_transform(image_size: int = 224):
    mean = (0.485, 0.456, 0.406)
    std = (0.228, 0.224, 0.225)
    return T.Compose([
        T.Resize(256),
        T.CenterCrop(image_size),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])


IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.228, 0.224, 0.225]).view(1, 3, 1, 1)


def unnormalize(x: torch.Tensor) -> torch.Tensor:
    mean = IMAGENET_MEAN.to(x.device)
    std = IMAGENET_STD.to(x.device)
    return torch.clamp(x * std + mean, 0.0, 1.0)


def normalize(x01: torch.Tensor) -> torch.Tensor:
    mean = IMAGENET_MEAN.to(x01.device)
    std = IMAGENET_STD.to(x01.device)
    return (torch.clamp(x01, 0.0, 1.0) - mean) / std


def load_rgb(path: Path) -> Image.Image:
    with Image.open(path) as img:
        return img.convert("RGB")


def resolve_image_path(row, img_dir: Path, image_col: str = "image_rel_path") -> Path:
    image_name = str(row[image_col])
    p = Path(image_name)

    if p.is_absolute():
        return p

    candidate = img_dir / p
    if candidate.exists():
        return candidate

    if image_name.endswith((".jpg", ".jpeg", ".png")):
        return img_dir / image_name

    return img_dir / f"{image_name}.jpg"


def get_cam_target_layer(model_raw: torch.nn.Module, target_block_index: int = -1):
    blocks = model_raw.backbone.blocks if hasattr(model_raw, "backbone") else model_raw.blocks
    n_blocks = len(blocks)

    if target_block_index < 0:
        resolved_index = n_blocks + target_block_index
    else:
        resolved_index = target_block_index

    if resolved_index < 0 or resolved_index >= n_blocks:
        raise ValueError(
            f"target_block_index={target_block_index} resolves to {resolved_index}, "
            f"but model has {n_blocks} blocks."
        )

    return blocks[resolved_index].norm1


def minmax_cam(cam_np: np.ndarray) -> np.ndarray:
    cam = np.asarray(cam_np).squeeze().astype(np.float32)
    cam = cam - np.nanmin(cam)
    denom = np.nanmax(cam) + 1e-8
    return cam / denom


def cam_to_tensor(cam_np: np.ndarray, device: str) -> torch.Tensor:
    return torch.from_numpy(minmax_cam(cam_np)).float().unsqueeze(0).to(device)


def resize_cam_to_image(cam: torch.Tensor, image_hw: tuple[int, int]) -> torch.Tensor:
    if cam.ndim == 2:
        cam = cam.unsqueeze(0)

    if cam.shape[-2:] != image_hw:
        cam = nn.functional.interpolate(
            cam.unsqueeze(1),
            size=image_hw,
            mode="bilinear",
            align_corners=False,
        ).squeeze(1)

    return cam


def mask_from_cam(cam: torch.Tensor, ratio: float) -> torch.Tensor:
    if ratio <= 0:
        return torch.zeros_like(cam, dtype=torch.bool)

    ratio = min(max(float(ratio), 0.0), 1.0)
    flat = cam.flatten()
    threshold = torch.quantile(flat, 1.0 - ratio)
    return cam >= threshold


def random_mask_like(cam: torch.Tensor, ratio: float, generator: torch.Generator | None = None) -> torch.Tensor:
    if ratio <= 0:
        return torch.zeros_like(cam, dtype=torch.bool)

    n = cam.numel()
    k = max(1, int(round(float(ratio) * n)))

    flat = torch.zeros(n, dtype=torch.bool, device=cam.device)
    perm = torch.randperm(n, device=cam.device, generator=generator)
    flat[perm[:k]] = True

    return flat.view_as(cam)


def perturb_image_soft(
    image_norm: torch.Tensor,
    mask_hw: torch.Tensor,
    perturbation_type: str,
    darken_factor: float = 0.45,
    noise_std: float = 0.18,
    generator: torch.Generator | None = None,
) -> torch.Tensor:
    x = unnormalize(image_norm).clone()

    mask = mask_hw.bool().to(x.device)
    mask3 = mask.unsqueeze(0).unsqueeze(0).expand_as(x)

    if perturbation_type == "darken":
        x[mask3] = x[mask3] * float(darken_factor)

    elif perturbation_type == "desaturate":
        r = x[:, 0:1]
        g = x[:, 1:2]
        b = x[:, 2:3]
        gray = 0.299 * r + 0.587 * g + 0.114 * b
        gray3 = gray.expand_as(x)
        x = torch.where(mask3, gray3, x)

    elif perturbation_type == "gaussian_noise":
        noise = torch.randn(x.shape, device=x.device, generator=generator) * float(noise_std)
        x_noisy = torch.clamp(x + noise, 0.0, 1.0)
        x = torch.where(mask3, x_noisy, x)

    else:
        raise ValueError(f"Unknown perturbation_type: {perturbation_type}")

    return normalize(x)

In [ ]:
df = pd.read_csv(CSV_PATH, low_memory=False)

required_cols = ["image_id", "image_rel_path", "gt_label", "label"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in {CSV_PATH}: {missing}")

if START_INDEX > 0:
    df = df.iloc[START_INDEX:].copy()

if NUM_SAMPLES is not None and int(NUM_SAMPLES) > 0:
    df = df.head(int(NUM_SAMPLES)).copy()

df = df.reset_index(drop=True)

print("Rows to evaluate:", len(df))
display(df[["image_id", "gt_label", "label", "image_rel_path"]].head())
display(df["gt_label"].value_counts())

In [ ]:
device = get_device(DEVICE)
print("Device:", device)

args_for_classes = SimpleNamespace(class_preset=CLASS_PRESET, class_names=CLASS_NAMES)
class_names = resolve_class_names(args_for_classes)
class_to_idx, idx_to_class = build_class_maps(class_names)

print("Class names:", class_names)

model_raw, model_info = load_panderm_finetuned_model(
    checkpoint_path=str(CHECKPOINT),
    num_classes=len(class_names),
    class_to_idx=class_to_idx,
    idx_to_class=idx_to_class,
    device=device,
    checkpoint_model_type=CHECKPOINT_MODEL_TYPE,
    use_seg_gate=False,
)

model = PanDermCAMWrapper(model_raw)
model.eval()

target_layer = get_cam_target_layer(model_raw, target_block_index=TARGET_BLOCK_INDEX)
transform = build_eval_transform()

print("Loaded model.")
print("Model info:", model_info)
print("Target block index:", TARGET_BLOCK_INDEX)

In [ ]:
def cache_path_for(image_id: str, method: str) -> Path:
    return CAM_CACHE_DIR / image_id / f"{method}.npy"


def load_cached_cams(image_id: str, methods: list[str]):
    cams = {}
    for method in methods:
        p = cache_path_for(image_id, method)
        if not p.exists():
            return None
        cams[method] = np.load(p)
    return cams


def save_cached_cams(image_id: str, cams: dict[str, np.ndarray]):
    out_dir = CAM_CACHE_DIR / image_id
    out_dir.mkdir(parents=True, exist_ok=True)

    for method, cam in cams.items():
        np.save(out_dir / f"{method}.npy", np.asarray(cam).astype(np.float32))


def determine_comparison_indices(probs: np.ndarray, gt_idx: int):
    sorted_idx = np.argsort(probs)[::-1]

    if COMPARE_MODE == "gt_topk_non_target":
        A_idx = int(gt_idx)
        comparison_categories = [
            int(i) for i in sorted_idx if int(i) != A_idx
        ][:max(1, int(TOPK_COMPARE))]

        if len(comparison_categories) == 0:
            raise ValueError("Could not find non-target comparison categories.")

        B_idx = comparison_categories[0]
        return A_idx, B_idx, comparison_categories

    if COMPARE_MODE == "pred_topk_non_target":
        A_idx = int(sorted_idx[0])
        comparison_categories = [
            int(i) for i in sorted_idx if int(i) != A_idx
        ][:max(1, int(TOPK_COMPARE))]

        B_idx = comparison_categories[0]
        return A_idx, B_idx, comparison_categories

    if COMPARE_MODE == "top2":
        A_idx = int(sorted_idx[0])
        B_idx = int(sorted_idx[1])
        return A_idx, B_idx, [B_idx]

    raise ValueError(f"Unsupported COMPARE_MODE: {COMPARE_MODE}")


def compute_or_load_cams(image_id: str, image_tensor: torch.Tensor, rgb_float: np.ndarray, gt_idx: int):
    requested_nonrandom = [m for m in METHODS if m != "random_region"]

    if SAVE_CAM_CACHE:
        cached = load_cached_cams(image_id, requested_nonrandom)
        if cached is not None:
            return cached

    with torch.no_grad():
        logits = model(image_tensor)
        probs = torch.softmax(logits, dim=1)[0].detach().cpu().numpy()

    A_idx, B_idx, comparison_categories = determine_comparison_indices(probs=probs, gt_idx=gt_idx)

    bundle = compute_cam_bundle(
        model=model,
        input_tensor=image_tensor,
        rgb_float=rgb_float,
        target_layer=target_layer,
        reshape_transform=vit_reshape_transform,
        method="finercam",
        A=A_idx,
        B=B_idx,
        comparison_categories=comparison_categories,
        alpha=ALPHA,
        include_extra_maps=False,
    )

    gradcam_diff_cam = bundle.get("cam_diff") if bundle.get("cam_diff") is not None else bundle.get("cam_gradcam_diff")

    cams = {
        "gradcam_target": bundle.get("cam_gradcam"),
        "gradcam_diff": gradcam_diff_cam,
        "finercam": bundle.get("cam_finercam"),
    }

    cams = {
        k: minmax_cam(v)
        for k, v in cams.items()
        if v is not None and k in requested_nonrandom
    }

    if SAVE_CAM_CACHE:
        save_cached_cams(image_id, cams)

    return cams

def random_mask_from_fixed_order(cam: torch.Tensor, ratio: float, perm: torch.Tensor) -> torch.Tensor:
    if ratio <= 0:
        return torch.zeros_like(cam, dtype=torch.bool)

    n = cam.numel()
    k = max(1, int(round(float(ratio) * n)))

    flat = torch.zeros(n, dtype=torch.bool, device=cam.device)
    flat[perm[:k]] = True

    return flat.view_as(cam)

In [ ]:
def evaluate_one_image_incremental(row, rng: torch.Generator):
    image_id = str(row["image_id"])
    gt_idx = int(row["label"])
    gt_name = str(row["gt_label"])

    image_path = resolve_image_path(row, IMG_DIR, image_col="image_rel_path")
    rgb = load_rgb(image_path)
    image_tensor = transform(rgb).unsqueeze(0).to(device)

    rgb_float = np.array(rgb).astype(np.float32) / 255.0
    rgb_float = cv2.resize(rgb_float, (224, 224), interpolation=cv2.INTER_LINEAR)

    with torch.no_grad():
        original_logits = model(image_tensor)
        original_probs_t = torch.softmax(original_logits, dim=1)[0]
        original_probs = original_probs_t.detach().cpu().numpy()

    pred_idx = int(np.argmax(original_probs))
    pred_name = class_names[pred_idx]
    pred_prob = float(original_probs[pred_idx])

    A_idx, B_idx, comparison_categories = determine_comparison_indices(
        probs=original_probs,
        gt_idx=gt_idx,
    )

    target_idx = int(A_idx)
    reference_idx = int(B_idx)

    target_name = class_names[target_idx]
    reference_name = class_names[reference_idx]

    original_target_conf = float(original_probs[target_idx])
    original_reference_conf = float(original_probs[reference_idx])

    cams = compute_or_load_cams(
        image_id=image_id,
        image_tensor=image_tensor,
        rgb_float=rgb_float,
        gt_idx=gt_idx,
    )

    if INCLUDE_RANDOM_BASELINE:
        cams["random_region"] = np.zeros((224, 224), dtype=np.float32)

    rows = []

    for method, cam_np in cams.items():
        if method == "random_region":
            cam_t = torch.zeros((1, 224, 224), dtype=torch.float32, device=device)
        else:
            cam_t = cam_to_tensor(cam_np, device=device)
            cam_t = resize_cam_to_image(cam_t, image_tensor.shape[-2:])

        cam_hw = cam_t[0]

        # Important:
        # For random_region, create ONE fixed random pixel order per image.
        # Then 10%, 20%, 30%, ... are nested and comparable.
        if method == "random_region":
            random_perm = torch.randperm(
                cam_hw.numel(),
                device=cam_hw.device,
                generator=rng,
            )
        else:
            random_perm = None

        for perturbation_type in PERTURBATION_TYPES:
            for ratio in PERTURBATION_RATIOS:
                if method == "random_region":
                    base_mask = random_mask_from_fixed_order(
                        cam_hw,
                        ratio=ratio,
                        perm=random_perm,
                    )
                else:
                    base_mask = mask_from_cam(cam_hw, ratio=ratio)

                perturbed = perturb_image_soft(
                    image_norm=image_tensor,
                    mask_hw=base_mask,
                    perturbation_type=perturbation_type,
                    darken_factor=DARKEN_FACTOR,
                    noise_std=NOISE_STD,
                    generator=rng,
                )

                with torch.no_grad():
                    pert_logits = model(perturbed)
                    pert_probs = torch.softmax(pert_logits, dim=1)[0].detach().cpu().numpy()

                pert_target_conf = float(pert_probs[target_idx])
                pert_reference_conf = float(pert_probs[reference_idx])

                target_drop = original_target_conf - pert_target_conf
                reference_drop = original_reference_conf - pert_reference_conf

                # Finer-CAM paper style:
                # RD = target drop - reference drop
                finer_cam_rd = target_drop - reference_drop

                pert_pred_idx = int(np.argmax(pert_probs))
                pert_pred_name = class_names[pert_pred_idx]
                pert_pred_prob = float(pert_probs[pert_pred_idx])
                pred_changed = bool(pert_pred_idx != pred_idx)

                rows.append({
                    "image_id": image_id,
                    "image_path": str(image_path),
                    "gt_idx": gt_idx,
                    "gt_name": gt_name,
                    "pred_idx": pred_idx,
                    "pred_name": pred_name,
                    "pred_prob": pred_prob,
                    "prediction_correct": bool(pred_idx == gt_idx),
                    "target_idx": target_idx,
                    "target_name": target_name,
                    "reference_idx": reference_idx,
                    "reference_name": reference_name,
                    "original_target_confidence": original_target_conf,
                    "original_reference_confidence": original_reference_conf,
                    "method": method,
                    "perturbation_type": perturbation_type,
                    "ratio": float(ratio),
                    "perturbed_target_confidence": pert_target_conf,
                    "perturbed_reference_confidence": pert_reference_conf,
                    "target_drop": float(target_drop),
                    "reference_drop": float(reference_drop),
                    "finer_cam_rd": float(finer_cam_rd),
                    "perturbed_pred_idx": pert_pred_idx,
                    "perturbed_pred_name": pert_pred_name,
                    "perturbed_pred_prob": pert_pred_prob,
                    "pred_changed": pred_changed,
                    "target_block_index": TARGET_BLOCK_INDEX,
                    "compare_mode": COMPARE_MODE,
                })

    return rows


all_rows = []
rng = torch.Generator(device=device)
rng.manual_seed(RANDOM_SEED)

for _, row in tqdm(df.iterrows(), total=len(df), desc="incremental soft perturbation"):
    try:
        all_rows.extend(evaluate_one_image_incremental(row, rng=rng))
    except Exception as e:
        image_id = str(row.get("image_id", "unknown"))
        print(f"[warn] failed image_id={image_id}: {e}")
        all_rows.append({
            "image_id": image_id,
            "status": "failed",
            "error": str(e),
        })

per_sample_df = pd.DataFrame(all_rows)

per_sample_path = OUT_ROOT / "incremental_soft_perturbation_per_sample.csv"
per_sample_df.to_csv(per_sample_path, index=False)

print("Saved:", per_sample_path)
print("Shape:", per_sample_df.shape)
display(per_sample_df.head())

In [ ]:
if "status" in per_sample_df.columns:
    ok_df = per_sample_df[per_sample_df["status"].fillna("ok") != "failed"].copy()
else:
    ok_df = per_sample_df.copy()

curve_summary = (
    ok_df
    .groupby(["method", "perturbation_type", "ratio"], dropna=False)
    .agg(
        n=("image_id", "count"),
        target_confidence_mean=("perturbed_target_confidence", "mean"),
        target_confidence_std=("perturbed_target_confidence", "std"),
        reference_confidence_mean=("perturbed_reference_confidence", "mean"),
        reference_confidence_std=("perturbed_reference_confidence", "std"),
        target_drop_mean=("target_drop", "mean"),
        reference_drop_mean=("reference_drop", "mean"),
        finer_cam_rd_mean=("finer_cam_rd", "mean"),
        pred_changed_rate=("pred_changed", "mean"),
        original_target_confidence_mean=("original_target_confidence", "mean"),
        original_reference_confidence_mean=("original_reference_confidence", "mean"),
    )
    .reset_index()
)

curve_summary_path = OUT_ROOT / "incremental_soft_perturbation_curves.csv"
curve_summary.to_csv(curve_summary_path, index=False)

print("Saved:", curve_summary_path)
display(curve_summary.head(20))

In [ ]:
method_display = {
    "gradcam_target": "Model focus",
    "gradcam_diff": "Class contrast",
    "finercam": "Refined contrast",
    "random_region": "Random region",
}

perturb_display = {
    "darken": "Darken region",
    "desaturate": "Remove color",
    "gaussian_noise": "Add visual noise",
}


def plot_confidence_curves(summary_df, perturbation_type: str, include_random: bool = True):
    sub = summary_df[summary_df["perturbation_type"] == perturbation_type].copy()

    if not include_random:
        sub = sub[sub["method"] != "random_region"].copy()

    if len(sub) == 0:
        print("No data for:", perturbation_type)
        return

    method_order = ["gradcam_target", "gradcam_diff", "finercam"]
    if include_random:
        method_order.append("random_region")

    plt.figure(figsize=(7, 5))

    for method in method_order:
        m = sub[sub["method"] == method].sort_values("ratio")
        if len(m) == 0:
            continue

        x = m["ratio"].values * 100
        y_target = m["target_confidence_mean"].values

        plt.plot(
            x,
            y_target,
            marker="o",
            linewidth=2,
            label=f"{method_display.get(method, method)} target",
        )

    plt.xlabel("Perturbed pixels (%)")
    plt.ylabel("Mean target class confidence")
    plt.title(f"Incremental soft perturbation: {perturb_display.get(perturbation_type, perturbation_type)}")
    plt.legend()
    plt.tight_layout()

    out_path = OUT_ROOT / f"curve_target_confidence_{perturbation_type}.png"
    plt.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.show()

    print("Saved:", out_path)


for ptype in PERTURBATION_TYPES:
    plot_confidence_curves(curve_summary, ptype, include_random=True)

In [ ]:
def plot_target_reference_curves(summary_df, perturbation_type: str, methods=("gradcam_target", "finercam")):
    sub = summary_df[
        (summary_df["perturbation_type"] == perturbation_type)
        & (summary_df["method"].isin(methods))
    ].copy()

    if len(sub) == 0:
        print("No data for:", perturbation_type)
        return

    plt.figure(figsize=(7, 5))

    for method in methods:
        m = sub[sub["method"] == method].sort_values("ratio")
        if len(m) == 0:
            continue

        x = m["ratio"].values * 100

        plt.plot(
            x,
            m["target_confidence_mean"].values,
            linewidth=2,
            label=f"{method_display.get(method, method)} target",
        )

        plt.plot(
            x,
            m["reference_confidence_mean"].values,
            linewidth=2,
            linestyle="--",
            label=f"{method_display.get(method, method)} reference",
        )

    plt.xlabel("Perturbed pixels (%)")
    plt.ylabel("Mean confidence")
    plt.title(f"Target vs reference confidence: {perturb_display.get(perturbation_type, perturbation_type)}")
    plt.legend()
    plt.tight_layout()

    out_path = OUT_ROOT / f"curve_target_vs_reference_{perturbation_type}.png"
    plt.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.show()

    print("Saved:", out_path)


for ptype in PERTURBATION_TYPES:
    plot_target_reference_curves(
        curve_summary,
        perturbation_type=ptype,
        methods=("gradcam_target", "finercam"),
    )

- solid lines: confidence for the true or target class
- dashed lines: confidence for the reference class

In [ ]:
def plot_rd_curves(summary_df, perturbation_type: str, include_random: bool = True):
    sub = summary_df[summary_df["perturbation_type"] == perturbation_type].copy()

    if not include_random:
        sub = sub[sub["method"] != "random_region"].copy()

    if len(sub) == 0:
        print("No data for:", perturbation_type)
        return

    method_order = ["gradcam_target", "gradcam_diff", "finercam"]
    if include_random:
        method_order.append("random_region")

    plt.figure(figsize=(7, 5))

    for method in method_order:
        m = sub[sub["method"] == method].sort_values("ratio")
        if len(m) == 0:
            continue

        x = m["ratio"].values * 100
        y = m["finer_cam_rd_mean"].values

        plt.plot(
            x,
            y,
            marker="o",
            linewidth=2,
            label=method_display.get(method, method),
        )

    plt.axhline(0, linestyle="--", linewidth=1)
    plt.xlabel("Perturbed pixels (%)")
    plt.ylabel("Relative confidence drop")
    plt.title(f"RD curve: {perturb_display.get(perturbation_type, perturbation_type)}")
    plt.legend()
    plt.tight_layout()

    out_path = OUT_ROOT / f"curve_rd_{perturbation_type}.png"
    plt.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.show()

    print("Saved:", out_path)


for ptype in PERTURBATION_TYPES:
    plot_rd_curves(curve_summary, ptype, include_random=True)

In [ ]:
# def trapezoid_auc(x, y):
#     return float(np.trapezoid(y, x=x))


# auc_rows = []

# for (method, perturbation_type), sub in curve_summary.groupby(["method", "perturbation_type"]):
#     sub = sub.sort_values("ratio")

#     x = sub["ratio"].values.astype(float)

#     target_conf_auc = trapezoid_auc(x, sub["target_confidence_mean"].values)
#     rd_auc = trapezoid_auc(x, sub["finer_cam_rd_mean"].values)

#     auc_rows.append({
#         "method": method,
#         "perturbation_type": perturbation_type,
#         "target_confidence_auc": target_conf_auc,
#         "rd_auc": rd_auc,
#         "mean_target_drop_at_30": float(
#             sub.loc[np.isclose(sub["ratio"], 0.3), "target_drop_mean"].mean()
#         ),
#         "mean_rd_at_30": float(
#             sub.loc[np.isclose(sub["ratio"], 0.3), "finer_cam_rd_mean"].mean()
#         ),
#     })

# auc_df = pd.DataFrame(auc_rows)

# auc_df["Method"] = auc_df["method"].map(method_display).fillna(auc_df["method"])
# auc_df["Perturbation"] = auc_df["perturbation_type"].map(perturb_display).fillna(auc_df["perturbation_type"])

# auc_out = OUT_ROOT / "incremental_soft_perturbation_auc_summary.csv"
# auc_df.to_csv(auc_out, index=False)

# print("Saved:", auc_out)
# display(auc_df[[
#     "Method",
#     "Perturbation",
#     "target_confidence_auc",
#     "rd_auc",
#     "mean_target_drop_at_30",
#     "mean_rd_at_30",
# ]])